In [41]:
from IPython.display import display, HTML
display(HTML("""
<style>
div.container{width:95% !important;}
div.cell.code_cell.rendered{width:95%;}
div.input_prompt{padding:0px;}
div.CodeMirror {font-family:Consolas; font-size:20pt;}
.inner_cell{font-size:20pt;}
div.text_cell_render pre code {font-size:20pt; line-height:30px;}
div.output {font-size:20pt; font-weight:bold;}
div.input {font-family:Consolas; font-size:20pt;}
div.prompt {min-width:70px;}
div#toc-wrapper{padding-top:120px;}
div.text_cell_render ul li{font-size:20pt;padding:5px;}
table.dataframe{font-size:20px;}
</style>
"""))

<b><font color="red" size="6">ch15. 데이터베이스 연동</font></b>
# 1절. SQLite 데이터 베이스 연결
- SQLite 데이터베이스는 별도의 DBMS없이 SQL을 이용해서 DB액세스할 수 있도록 만든 간단한 디스크기반 DB제공
- C라이브러리
- SQLite는 프로토타입을 만들 때 사용
- 프로젝트단계 : 분석  ->  설계  ->  구현  ->  테스트  ->  고객에게 배포  ->  유지보수
          -      프로토타입(SQLite) 시제품(구현후 반양산직전) 완제품(Oracle, MySQL, MariaDB, PostgreSQL, MSSQL, 아마존auroraDB, ...)

- [DB Browser for SQLite](https://sqlitebrowser.org/)에서 "DB Browser for SQLite - .zip (no installer) for 64-bit Windows" 다운로드후 압축 풀기

## 1.1 SQLite browser 설치 및 sqlite3패키지 load

In [5]:
import sqlite3
sqlite3.sqlite_version

'3.40.1'

In [2]:
import pandas as pd
pd.__version__

'1.5.3'

## 1.2 데이터베이스 연결
- 데이터베이스연결 객체 -> 커서객체(SQL전송 및 결과 받는 객체) -> 원하는 로직 수행 -> 커서객체 해제 -> DB연결객체 해제(close)
- SQLite로 DB 연결객체 생성시, DB파일이 있으면 연결, DB파일이 없으면 빈 DB파일 생성

In [12]:
# DB연결 (여기서 에러가 날 경우 VC_redist.x64.exe 설치)
conn = sqlite3.connect('data/ch15_example.db')
conn

In [13]:
# 커서 객체 생성 : 커서는 SQL문 실행시키고, 결과를 받는 객체
cursor = conn.cursor()
cursor

In [8]:
cursor.execute('''
    CREATE TABLE MEMBER (
        NAME TEXT,
        AGE  INT,
        EMAIL TEXT 
    )
''')

In [9]:
cursor.execute('DROP TABLE MEMBER')

In [15]:
sql = 'INSERT INTO MEMBER VALUES (\'마길동\', 25, \'h@h.com\')'
cursor.execute(sql) # sql 전송
print('insert, update, delete문의 수행 결과 행수 :', cursor.rowcount)
sql = "INSERT INTO MEMBER VALUES ('신길동', 30, 's@h.com')"
cursor.execute(sql) 
print('수행 결과 행수 :', cursor.rowcount)
sql = "INSERT INTO MEMBER VALUES ('신림동', 35, 'sil@h.com')"
cursor.execute(sql) 
print('수행 결과 행수 :', cursor.rowcount)

insert, update, delete문의 수행 결과 행수 : 1
수행 결과 행수 : 1
수행 결과 행수 : 1


In [16]:
conn.commit() # 反. conn.rollback()DML에서만 commit이나 rollback

In [17]:
cursor.execute("SELECT * FROM MEMBER ORDER BY AGE") # SELECT sql문 전송

In [18]:
# INSERT, UPDATE, DELETE문 실행결과 : cursor.rowcount
# SELECT문 실행결과를 받는 함수들
    # fetchone() : 결과를 한행씩 받을 때 (튜플)
    # fetchall() : 결과를 모두 받을 때 (튜플 list)
    # fetchmany(n) : 결과를 n행 받을 때(튜플 list)
cursor.fetchall()

[('홍길동', 25, 'h@h.com'),
 ('마길동', 25, 'h@h.com'),
 ('신길동', 30, 's@h.com'),
 ('신림동', 35, 'sil@h.com')]

In [19]:
cursor.fetchall() # 한번 소요된 cursor 객체는 다시 fetch할 수 없음

[]

In [20]:
cursor.execute("SELECT * FROM MEMBER ORDER BY AGE")
members = cursor.fetchall()
members

[('홍길동', 25, 'h@h.com'),
 ('마길동', 25, 'h@h.com'),
 ('신길동', 30, 's@h.com'),
 ('신림동', 35, 'sil@h.com')]

In [22]:
# 한줄씩 읽어서 dict list에 append
cursor.execute("SELECT * FROM MEMBER ORDER BY AGE")
members = []
while True:
    member = cursor.fetchone() # SQL문 수행결과를 한줄 가져오기
    if member is None:
        break
    members.append({'name':member[0], 'age':member[1], 'email':member[2]})
members

[{'name': '홍길동', 'age': 25, 'email': 'h@h.com'},
 {'name': '마길동', 'age': 25, 'email': 'h@h.com'},
 {'name': '신길동', 'age': 30, 'email': 's@h.com'},
 {'name': '신림동', 'age': 35, 'email': 'sil@h.com'}]

In [23]:
class Member:
    'Member 테이블의 내용을 받은 객체 타입'
    def __init__(self, name, age, email):
        self.name = name
        self.age  = age
        self.email = email
    def __str__(self):
        return "{}\t{}\t{}".format(self.name, self.age, self.email)
m = Member('홍길동', 25, 'h@h.com')
print(m)

홍길동	25	h@h.com


In [24]:
dbmember = ('홍길동', 25, 'h@h.com')
m = Member(*dbmember)
print(m)

홍길동	25	h@h.com


In [30]:
# 한줄씩 읽어서 객체list에 append
cursor.execute("SELECT * FROM MEMBER ORDER BY AGE")
members = []
while True:
    dbmember = cursor.fetchone()
    if dbmember is None:
        break
    member = Member(*dbmember)
    members.append(member)
for mem in members:
    print(mem)

홍길동	25	h@h.com
마길동	25	h@h.com
신길동	30	s@h.com
신림동	35	sil@h.com


In [31]:
# 최상위 n행 읽어오기
cursor.execute("SELECT * FROM MEMBER ORDER BY AGE")
members = cursor.fetchmany(2)
members

[('홍길동', 25, 'h@h.com'), ('마길동', 25, 'h@h.com')]

In [33]:
cursor.close()
conn.close()

## 1.3 SQL구문에 파라미터 사용하기
- qmark(DB에 따라 불가한 경우가 있음)
- named(추천)

In [34]:
conn = sqlite3.connect('data/ch15_example.db')
cursor = conn.cursor()
cursor.execute("SELECT * FROM MEMBER WHERE NAME IN ('홍길동','신길동')")
cursor.fetchall()

[('홍길동', 25, 'h@h.com'), ('신길동', 30, 's@h.com')]

In [36]:
# 파라미터 사용하기 : qmark 방법 이용
name1 = input('검색할 이름1 :')
name2 = input('검색할 이름2 :')
# cursor.execute(f"SELECT * FROM MEMBER WHERE NAME IN ('{name1}', '{name2}')")
cursor.execute("SELECT * FROM MEMBER WHERE NAME IN (?, ?)", (name1, name2))
cursor.fetchall()

검색할 이름1 :신길동
검색할 이름2 :신림동


[('신길동', 30, 's@h.com'), ('신림동', 35, 'sil@h.com')]

In [37]:
# 파라미터 사용하기 : named 방법 이용
name1 = input('검색할 이름1 :')
name2 = input('검색할 이름2 :')
# cursor.execute(f"SELECT * FROM MEMBER WHERE NAME IN ('{name1}', '{name2}')")
cursor.execute("SELECT * FROM MEMBER WHERE NAME IN (:name1, :name2)", {'name1':name1,
                                                                      'name2':name2})
cursor.fetchall()

검색할 이름1 :마마마
검색할 이름2 :가가가


[]

In [39]:
# 파라미터 이용하기 : named 방법
name = input('회원가입할 이름은?')
try:
    age = int(input('나이는?(꼭 숫자로)'))
except:
    print('유효하지 않은 나이를 입력할 경우 1세로 초기화합니다')
    age = 1
email = input('이메일은?')
cursor.execute("INSERT INTO MEMBER VALUES (:name, :age, :email)", 
              {'name':name, 'age':age, 'email':email}) # sql 전송
conn.commit()
print('수행 결과 행수 :', cursor.rowcount)

회원가입할 이름은?박길동
나이는?(꼭 숫자로)이팔청춘
유효하지 않은 나이를 입력할 경우 1세로 초기화합니다
이메일은?l@l.com
수행 결과 행수 : 1


In [40]:
cursor.close()
conn.close()

# 2절. 오라클 데이터베이스 연결
- pip install cx_oracle(11g까지), pip install oracledb(12버전부터)

In [42]:
import cx_Oracle
cx_Oracle.__version__

'8.3.0'

In [45]:
# conn 얻어오는 방법1
oracle_dsn = cx_Oracle.makedsn(host="localhost", port=1521, sid='xe')
# conn = cx_Oracle.connect(user='scott', password='tiger', dsn=oracle_dsn)
conn = cx_Oracle.connect('scott', 'tiger', oracle_dsn)
conn.close()

In [49]:
# conn 얻어오는 방법 2 : (여기서 에러가 날 경우 VC_redist.x64.exe 설치)
#conn = cx_Oracle.connect(user='scott', password='tiger', dsn='localhost:1521/xe')
conn = cx_Oracle.connect('scott', 'tiger', 'localhost:1521/xe')
conn

<cx_Oracle.Connection to scott@localhost:1521/xe>

In [51]:
# cursor 객체 생성하고 sql문 전송&결과 받기
cursor = conn.cursor()
sql = "SELECT EMPNO NO, ENAME, JOB, MGR, HIREDATE, SAL, COMM, DEPTNO FROM EMP"
cursor.execute(sql)
emps = cursor.fetchall()

In [52]:
for emp in emps:
    print(emp)

(7369, 'SMITH', 'CLERK', 7902, datetime.datetime(1980, 12, 17, 0, 0), 800.0, None, 20)
(7499, 'ALLEN', 'SALESMAN', 7698, datetime.datetime(1981, 2, 20, 0, 0), 1600.0, 300.0, 30)
(7521, 'WARD', 'SALESMAN', 7698, datetime.datetime(1981, 2, 22, 0, 0), 1250.0, 500.0, 30)
(7566, 'JONES', 'MANAGER', 7839, datetime.datetime(1981, 4, 2, 0, 0), 2975.0, None, 20)
(7654, 'MARTIN', 'SALESMAN', 7698, datetime.datetime(1981, 9, 28, 0, 0), 1250.0, 1400.0, 30)
(7698, 'BLAKE', 'MANAGER', 7839, datetime.datetime(1981, 5, 1, 0, 0), 2850.0, None, 30)
(7782, 'CLARK', 'MANAGER', 7839, datetime.datetime(1981, 6, 9, 0, 0), 2450.0, None, 10)
(7788, 'SCOTT', 'ANALYST', 7566, datetime.datetime(1982, 12, 9, 0, 0), 3000.0, None, 20)
(7839, 'KING', 'PRESIDENT', None, datetime.datetime(1981, 11, 17, 0, 0), 5000.0, None, 10)
(7844, 'TURNER', 'SALESMAN', 7698, datetime.datetime(1981, 9, 8, 0, 0), 1500.0, 0.0, 30)
(7876, 'ADAMS', 'CLERK', 7788, datetime.datetime(1983, 1, 12, 0, 0), 1100.0, None, 20)
(7900, 'JAMES', 'CL